# Notebook 3 — The dataset

Turns Notebook 2's lists into the texts the model reads, splits the releases, and writes the dataset
with a fixed schema. **One repository per run**: run cell 0 once, pick the repository in cell 1, then
run the rest.

| view | one text = | from |
|---|---|---|
| code | the file's **skeleton** (method bodies replaced by `{ … }`), or **one method** in full | the file at release t, read from git, parsed with tree-sitter-java |
| history | **one commit**: its message, then this file's diff | Notebook 2's commit list, the message from git, the diff from `diffs_<repo>.jsonl` |
| discourse | **one** issue body, PR body, comment or review | Notebook 2's issue/PR links and texts |

**Rules.**
- Nothing at or after release t is evidence: history comes from release t's own git history, and a
  discourse text counts only if it was written before t's cutoff.
- Bot commits and bot-written texts are left out (Notebook 2 flagged them).
- Split by release date, per repository: the oldest 50% of releases **train**, the next 20%
  **validation**, the newest 30% **test**. A release whose label became known only after the next part
  began is **left out**.

**Writes** (to `outputs/notebook_3_cbert/<repo>/`), for the picked repository:

| file | one line per | columns |
|---|---|---|
| `rows_<repo>.parquet` | (file, release) | row_id, repo, release, release_pos, path, cutoff_ts, label, split, code_ids, history_ids, discourse_ids |
| `code_texts_<repo>.parquet` | code text | code_id, kind (skeleton, method, file), text |
| `history_texts_<repo>.parquet` | (commit, file) | history_id, sha, commit_ts, text |
| `discourse_texts_<repo>.parquet` | discourse text | discourse_id, kind, ts, text |
| `split_<repo>.csv` | release | release, release_pos, cutoff, label_known, part |
| `manifest_<repo>.json` | file | lines and SHA-256 of every file above, package versions, and the report numbers |

`code_ids` lists the skeleton first, then the methods in file order. `history_ids` and
`discourse_ids` list the newest first. A row with no history or no discourse has an empty list.

| cell | what it does |
|---|---|
| 0 | install the packages (once, then restart the kernel) |
| 1 | pick the repository |
| 1a | the mined inputs, from Zenodo |
| 2 | setup and load Notebook 2's tables |
| 3 | the split |
| 4–5 | code texts |
| 6 | history texts |
| 7 | discourse texts |
| 8 | check, write, manifest |
| 9 | summary of all five repositories, once all five have run |

Run Notebook 2 first: this notebook reads its tables from `outputs/notebook_2_cbert/`.

### Cell 0 — Install (once)
Installs the four packages this notebook needs into the Python environment running this notebook.
The two tree-sitter packages are pinned to fixed versions, so the skeleton and method texts come out
the same on every machine. Run this cell once, **restart the kernel**, then start from cell 1.

In [11]:
%pip install tree-sitter==0.26.0 tree-sitter-java==0.23.5 pyarrow ipywidgets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


### Cell 1 — Pick the repository
Choose one from the list, then run cell 1a onwards. To run another repository, pick it here and run
cell 1a onwards again.

In [3]:
REPOS = ["checkstyle", "commons-lang", "dubbo", "junit-framework", "questdb"]
try:
    import ipywidgets as widgets
    from IPython.display import display
    PICK = widgets.Dropdown(options=REPOS, value="questdb", description="repository")
    display(PICK)
except ImportError:
    PICK = None
    print("ipywidgets is not installed: set REPO by hand in cell 2")

Dropdown(description='repository', index=4, options=('checkstyle', 'commons-lang', 'dubbo', 'junit-framework',…

### Cell 1a — The mined inputs, from Zenodo
Notebooks 0–3 read what was mined from the five repos: the release grids, Quarry's outputs and a git
clone of each repo. They are published on Zenodo, [10.5281/zenodo.24182007](https://doi.org/10.5281/zenodo.24182007),
as one zip and one git bundle per repo.

For the picked repo this cell downloads only what is missing from `workspace_data/`:
- the zip, checked against the record's `SHA256SUMS.txt` and unpacked into `workspace_data/`, every file
  checked against the zip's `MANIFEST_<repo>.json`;
- the bundle, checked the same way and restored as the clone in `workspace_data/repos/<repo>/`.

A workspace that already holds the files and clones (for example one that ran Quarry itself) downloads
nothing. The same cell opens Notebooks 0, 1, 2 and 3, so any of them can be run first. The first run
downloads about 2.5 GB.

In [ ]:
import os, json, hashlib, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

# The workspace Quarry uses: QUARRY_HOME if set, otherwise workspace_data beside the codebase.
WORKSPACE = Path(os.environ["QUARRY_HOME"]) if os.environ.get("QUARRY_HOME") \
    else Path.cwd().parent / "workspace_data"
MINED_RECORD = "24182007"                         # Zenodo record of the mined inputs
MINED_REPOS = [PICK.value if PICK is not None else "commons-lang"]
NEEDED = ["versions/rel_hist_{r}.json", "outputs/{r}/intervals_{r}.json", "outputs/{r}/refactorings_{r}.jsonl",
          "outputs/{r}/ledger_history_{r}.jsonl", "outputs/{r}/diffs_{r}.jsonl", "outputs/{r}/links_{r}.csv"]


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()


def have_mined_files(repo):
    """Every file the notebooks need, in outputs/<repo>/ or outputs/ (both layouts are read)."""
    def there(rel):
        p = WORKSPACE / rel.format(r=repo)
        return p.exists() or (WORKSPACE / "outputs" / p.name).exists()
    return all(there(x) for x in NEEDED)


def download(name, dest, sums):
    """One file of the record, checked against SHA256SUMS.txt before it is kept."""
    tmp = dest.with_name(dest.name + ".part")
    print(f"   downloading {name}")
    urllib.request.urlretrieve(f"https://zenodo.org/records/{MINED_RECORD}/files/{name}?download=1", tmp)
    if sha256_of(tmp) != sums[name]:
        tmp.unlink()
        raise SystemExit(f"{name}: SHA-256 differs from the record's SHA256SUMS.txt; download it again")
    tmp.replace(dest)


def fetch_mined(repos):
    downloads = WORKSPACE / "downloads"
    downloads.mkdir(parents=True, exist_ok=True)
    sums = None
    for repo in repos:
        clone = WORKSPACE / "repos" / repo
        need_files, need_clone = not have_mined_files(repo), not (clone / ".git").exists()
        if not (need_files or need_clone):
            print(f"{repo:<18}mined files and clone already in {WORKSPACE}")
            continue
        if sums is None:
            urllib.request.urlretrieve(f"https://zenodo.org/records/{MINED_RECORD}/files/SHA256SUMS.txt?download=1",
                                       downloads / "SHA256SUMS.txt")
            sums = {n: h for h, n in (l.split() for l in
                    (downloads / "SHA256SUMS.txt").read_text(encoding="utf-8").splitlines() if l.strip())}
        if need_files:
            z = downloads / f"{repo}_mined.zip"
            download(z.name, z, sums)
            with zipfile.ZipFile(z) as zf:
                manifest = json.loads(zf.read(f"MANIFEST_{repo}.json"))
                for f in manifest["files"]:
                    out = WORKSPACE / f["file"]
                    out.parent.mkdir(parents=True, exist_ok=True)
                    with zf.open(f["file"]) as src, open(out, "wb") as dst:
                        shutil.copyfileobj(src, dst, 1 << 24)
                    if sha256_of(out) != f["sha256"]:
                        raise SystemExit(f"{f['file']}: SHA-256 differs from MANIFEST_{repo}.json")
                (WORKSPACE / "versions" / f"MANIFEST_{repo}.json").write_bytes(zf.read(f"MANIFEST_{repo}.json"))
            z.unlink()
            print(f"{repo:<18}{len(manifest['files'])} mined files unpacked and checked")
        if need_clone:
            b = downloads / f"{repo}.bundle"
            download(b.name, b, sums)
            clone.mkdir(parents=True, exist_ok=True)
            for args in (["clone", "--mirror", "--quiet", str(b.resolve()), str(clone / ".git")],
                         ["-C", str(clone), "config", "--bool", "core.bare", "false"],
                         ["-C", str(clone), "-c", "core.longpaths=true", "reset", "--hard", "--quiet"]):
                r = subprocess.run(["git", *args], capture_output=True, text=True)
                if r.returncode != 0:
                    raise SystemExit(f"git {' '.join(args[:2])} failed for {repo}: {r.stderr.strip()[:300]}")
            b.unlink()
            print(f"{repo:<18}clone restored from {b.name}")


fetch_mined(MINED_REPOS)

### Cell 2 — Setup, and Notebook 2's tables
Paths, helpers, and the seven tables Notebook 2 wrote for this repository. The cell stops if a table
is missing or Notebook 2's manifest does not list it with the same line count.

In [4]:
import os, re, json, hashlib, subprocess, collections
from pathlib import Path

import numpy as np
import pandas as pd
import tree_sitter_java
from tree_sitter import Language, Parser

pd.set_option("display.width", 250, "display.max_columns", 40, "display.max_rows", 200)

REPO = PICK.value if PICK is not None else "commons-lang"

WORKSPACE = Path(os.environ["QUARRY_HOME"]) if os.environ.get("QUARRY_HOME") \
    else Path.cwd().parent / "workspace_data"
OUTPUTS, CLONES = WORKSPACE / "outputs", WORKSPACE / "repos"
NB2 = OUTPUTS / "notebook_2_cbert"
OUT = OUTPUTS / "notebook_3_cbert" / REPO                # one folder per repository
OUT.mkdir(parents=True, exist_ok=True)

SPLIT = {"train": 0.5, "validation": 0.2, "test": 0.3}      # by release date, oldest first
JAVA = Parser(Language(tree_sitter_java.language()))


def git(*args, stdin=None):
    """Run git in the clone; stop loudly if it fails rather than return an empty answer."""
    r = subprocess.run(["git", "-c", "core.quotepath=off", *args], cwd=CLONES / REPO,
                       input=stdin, capture_output=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args[:3])} failed: {r.stderr.decode('utf-8', 'replace')[:300]}")
    return r.stdout


def table(name):
    for ext in (".parquet", ".csv.gz", ".csv"):
        p = NB2 / f"{name}_{REPO}{ext}"
        if p.exists():
            return pd.read_parquet(p) if ext == ".parquet" else pd.read_csv(p)
    raise SystemExit(f"{name}_{REPO} not found in {NB2} -- run Notebook 2 first")


ROWS, RELEASES = table("rows"), table("releases")
HC, MSG = table("history_commits"), table("commit_messages")
ITEMS, COMMENTS, LINKS = table("text_items"), table("text_comments"), table("history_text")

listed = {f["table"]: f["lines"] for f in json.loads((NB2 / "manifest.json").read_text())["files"]
          if f["repo"] == REPO}
for name, df in [("rows", ROWS), ("releases", RELEASES), ("history_commits", HC), ("commit_messages", MSG),
                 ("text_items", ITEMS), ("text_comments", COMMENTS), ("history_text", LINKS)]:
    if listed.get(name) != len(df):
        raise SystemExit(f"{name}: {len(df):,} lines, Notebook 2's manifest says {listed.get(name)}")

ROWS["cutoff_ts"] = ROWS.release_pos.map(RELEASES.set_index("release_pos").cutoff_ts)
print(f"repository {REPO}\nwriting to {OUT}\n")
print(f"rows {len(ROWS):,}   refactored {int(ROWS.label.sum()):,}   commit lines {len(HC):,}   "
      f"issue/PR links {len(LINKS):,}   issues+PRs {len(ITEMS):,}   comments+reviews {len(COMMENTS):,}")

repository questdb
writing to E:\Thesis Project\Smell Ranker\workspace_data\outputs\notebook_3_cbert

rows 218,893   refactored 11,365   commit lines 2,921,802   issue/PR links 2,322,773   issues+PRs 6,376   comments+reviews 29,732


### Cell 3 — The split
The labelled releases are sorted by date (their cutoff). The oldest 50% are **train**, the next 20%
**validation**, the newest 30% **test**; releases sharing a date with the first release of a part go to
that part.

A release's label covers the work until its next release came out, so it is **known** only on the day
its last child release appeared. A train release whose label was known only after validation began,
or a validation release whose label was known only after test began, would learn from the later
period. It is **left out**: never trained on, tuned on or tested on.

In [14]:
kids = RELEASES.dropna(subset=["parent_pos"]).astype({"parent_pos": int})
known = kids.groupby("parent_pos").cutoff_ts.max()              # the day each label became known

S = RELEASES[RELEASES.release_pos.isin(ROWS.release_pos)][["release", "release_pos", "cutoff_ts", "cutoff"]]
S = S.sort_values(["cutoff_ts", "release_pos"]).reset_index(drop=True)
S["label_known_ts"] = S.release_pos.map(known)
if S.label_known_ts.isna().any():
    raise SystemExit("a release with rows has no child release, so its label has no end")

n = len(S)
val_start = S.cutoff_ts.iat[int(round(SPLIT["train"] * n))]
test_start = S.cutoff_ts.iat[int(round((SPLIT["train"] + SPLIT["validation"]) * n))]
S["part"] = np.where(S.cutoff_ts < val_start, "train", np.where(S.cutoff_ts < test_start, "validation", "test"))
S.loc[(S.part == "train") & (S.label_known_ts > val_start), "part"] = "left out"
S.loc[(S.part == "validation") & (S.label_known_ts > test_start), "part"] = "left out"
S["label_known"] = pd.to_datetime(S.label_known_ts, unit="s", utc=True).dt.strftime("%Y-%m-%d %H:%M")

ROWS["split"] = ROWS.release_pos.map(dict(zip(S.release_pos, S.part)))
REPORT = {"repo": REPO}
for part in ("train", "validation", "test", "left out"):
    m = ROWS.split == part
    REPORT[f"{part} releases"] = int((S.part == part).sum())
    REPORT[f"{part} rows"] = int(m.sum())
    REPORT[f"{part} refactored"] = int(ROWS.label[m].sum())
print(pd.Series(REPORT).to_string())
for _, r in S[S.part == "left out"].iterrows():
    print(f"left out {r.release}: released {r.cutoff[:10]}, label known {r.label_known[:10]}")

repo                     questdb
train releases                55
train rows                 67067
train refactored            5253
validation releases           23
validation rows            48987
validation refactored       2095
test releases                 34
test rows                  99069
test refactored             3764
left out releases              2
left out rows               3770
left out refactored          253
left out 6.7: released 2023-01-20, label known 2023-08-22
left out 7.0.1: released 2023-02-27, label known 2023-04-05


### Cell 4 — Which file version each row reads
`git ls-tree` lists every file's **blob** (its exact content) at each release. A file that did not
change between releases has the same blob, so it is read and parsed only once.

In [15]:
blob_of = {}                                                    # row_id -> blob
for pos, g in ROWS.groupby("release_pos"):
    sha = RELEASES.set_index("release_pos").sha[pos]
    tree = {}
    for entry in git("ls-tree", "-r", "-z", sha).split(b"\0"):
        if entry:
            meta, path = entry.split(b"\t", 1)
            tree[path.decode("utf-8", "replace")] = meta.split()[2].decode()
    for rid, path in zip(g.row_id, g.path):
        blob_of[rid] = tree.get(path)
missing = [r for r, b in blob_of.items() if b is None]
if missing:
    raise SystemExit(f"{len(missing)} rows' files are not in their release's tree, e.g. {missing[:3]}")
ROWS["blob"] = ROWS.row_id.map(blob_of)
BLOBS = sorted(set(blob_of.values()))
print(f"{len(ROWS):,} rows read {len(BLOBS):,} distinct file versions")

218,893 rows read 37,860 distinct file versions


### Cell 5 — Code texts: the skeleton and the methods
Each distinct file version is parsed with tree-sitter-java and gives:
- **one skeleton**: the file with every method and constructor body replaced by `{ … }`, and the
  package line, imports and comments removed. It keeps the type declarations, fields, and every
  method's signature.
- **one text per method or constructor that has a body**: its full source, with its Javadoc when it
  has one. Methods of anonymous classes stay inside their enclosing method.

A file that does not parse cleanly is kept whole as one text of kind `file`, and counted. Each text's
id is the SHA-1 of the text, so identical texts are stored once.

In [16]:
METHODS = {"method_declaration", "constructor_declaration", "compact_constructor_declaration"}
DROP = {"package_declaration", "import_declaration", "line_comment", "block_comment"}


def read_blobs(blobs):
    """blob -> bytes, through one git process per 2,000 blobs."""
    out = {}
    for i in range(0, len(blobs), 2000):
        part = blobs[i:i + 2000]
        raw, j = git("cat-file", "--batch", stdin="".join(b + "\n" for b in part).encode()), 0
        for b in part:
            nl = raw.index(b"\n", j)
            size = int(raw[j:nl].split()[2])
            out[b] = raw[nl + 1: nl + 1 + size]
            j = nl + 1 + size + 1
    return out


def code_texts(src):
    """(skeleton, [method texts]) for one file, or None if it does not parse cleanly."""
    tree = JAVA.parse(src)
    if tree.root_node.has_error:
        return None
    cuts, methods = [], []                           # cuts: (start, end, replacement)

    def visit(node):
        if node.type in METHODS and node.child_by_field_name("body") is not None:
            body = node.child_by_field_name("body")
            cuts.append((body.start_byte, body.end_byte, b"{ \xe2\x80\xa6 }"))
            doc = node.prev_sibling
            start = doc.start_byte if doc is not None and doc.type == "block_comment" \
                and src[doc.start_byte:doc.start_byte + 3] == b"/**" else node.start_byte
            methods.append(src[start:node.end_byte].decode("utf-8", "replace"))
            for c in node.children:                  # the signature may hold comments; the body is cut whole
                if c.id != body.id:
                    visit(c)
            return
        if node.type in DROP:
            cuts.append((node.start_byte, node.end_byte, b""))
            return
        for c in node.children:
            visit(c)

    visit(tree.root_node)
    parts, at = [], 0
    for s, e, rep in sorted(cuts):
        parts += [src[at:s], rep]
        at = e
    parts.append(src[at:])
    lines = [l.rstrip() for l in b"".join(parts).decode("utf-8", "replace").splitlines()]
    skeleton = "\n".join(l for l in lines if l.strip())
    return skeleton, methods


def cid(text):
    return hashlib.sha1(text.encode("utf-8")).hexdigest()


CODE, CODE_IDS, failed = {}, {}, 0                  # code_id -> (kind, text); blob -> [code_ids]
for b, src in read_blobs(BLOBS).items():
    got = code_texts(src)
    if got is None:
        failed += 1
        text = src.decode("utf-8", "replace")
        CODE[cid(text)] = ("file", text)
        CODE_IDS[b] = [cid(text)]
        continue
    skeleton, methods = got
    if not skeleton.strip():                       # nothing outside comments: keep the file whole
        text = src.decode("utf-8", "replace")
        CODE[cid(text)] = ("file", text)
        CODE_IDS[b] = [cid(text)]
        failed += 1
        continue
    ids = [cid(skeleton)]
    CODE[ids[0]] = ("skeleton", skeleton)
    for m in methods:
        CODE[cid(m)] = ("method", m)
        ids.append(cid(m))
    CODE_IDS[b] = ids

ROWS["code_ids"] = ROWS.blob.map(CODE_IDS)
per_row = ROWS.code_ids.map(len) - 1
REPORT.update({"file versions": len(BLOBS), "file versions not parsed (kept whole)": failed,
               "distinct code texts": len(CODE)})
print(f"{len(BLOBS):,} file versions   not parsed cleanly {failed:,} ({failed / max(1, len(BLOBS)):.2%})   "
      f"distinct code texts {len(CODE):,}")
print(f"methods per row: median {per_row.median():.0f}   90th percentile {per_row.quantile(.9):.0f}   "
      f"maximum {per_row.max():,}")

37,860 file versions   not parsed cleanly 0 (0.00%)   distinct code texts 113,077
methods per row: median 5   90th percentile 23   maximum 949


### Cell 6 — History texts: one per (commit, file)
Every non-bot commit in a row's history becomes one text, in the replication's markers:

```
<COMMIT_MESSAGE>message</COMMIT_MESSAGE>
<FILE>path in that commit</FILE>
<REMOVED>removed lines</REMOVED>
<ADDED>added lines</ADDED>
```

The message comes from git (Notebook 2). The removed and added lines are this file's lines in the
commit's record in `diffs_<repo>.jsonl`, written by Quarry's diff stage, blank lines dropped. That stage
runs without rename detection, so in the commit that renames a file the new name's whole content shows
as added. A text keeps its message only, and is counted, in two cases: the commit's record has no
non-blank changed line for this file, or the diffs file holds no record of the commit. The second
happens because the diff stage lists its commits with `git log <universe> -- '*.java'`, and git's
history simplification skips the commits on one side of a merge whose Java files match the other
parent. The same (commit, file) text is stored
once, however many releases' rows use it.

In [17]:
diffs_path = next((p for p in (OUTPUTS / REPO / f"diffs_{REPO}.jsonl", OUTPUTS / f"diffs_{REPO}.jsonl")
                   if p.exists()), None)
if diffs_path is None:
    raise SystemExit(f"diffs_{REPO}.jsonl not found in {OUTPUTS / REPO} or {OUTPUTS}")

bots = set(MSG.sha[MSG.bot.astype(bool)])
H = HC[~HC.sha.isin(bots)]
want = collections.defaultdict(set)                             # sha -> paths needed from its diff
for s, p in zip(H.sha, H.path_in_commit):
    want[s].add(p)

lines = {}                                                      # (sha, path) -> (removed, added)
seen = set()
with open(diffs_path, encoding="utf-8") as f:
    for line in f:
        m = re.search(r'"sha1":\s*"([0-9a-f]{40})"', line[:400])   # cheap test before parsing
        if m and m.group(1) not in want:
            continue
        rec = json.loads(line)
        if rec.get("sha1") not in want:
            continue
        if rec.get("status") != "success":
            continue
        seen.add(rec["sha1"])
        for fl in rec.get("files") or []:
            if fl.get("file") in want[rec["sha1"]]:
                lines[(rec["sha1"], fl["file"])] = ([l for l in fl.get("removed") or [] if l.strip()],
                                                    [l for l in fl.get("added") or [] if l.strip()])

message = dict(zip(MSG.sha, MSG.message.fillna("")))
commit_ts = dict(zip(MSG.sha, MSG.ts))
HIST, how = {}, collections.Counter()                            # history_id -> (sha, ts, text)
for s, p in set(zip(H.sha, H.path_in_commit)):
    text = f"<COMMIT_MESSAGE>{message[s].strip()}</COMMIT_MESSAGE>\n<FILE>{p}</FILE>"
    removed, added = lines.get((s, p), ([], []))
    if removed:
        text += "\n<REMOVED>" + "\n".join(removed) + "</REMOVED>"
    if added:
        text += "\n<ADDED>" + "\n".join(added) + "</ADDED>"
    how["message + diff" if removed or added else
        "message only: commit not in the diffs file" if s not in seen else
        "message only: no changed lines for this file"] += 1
    HIST[f"{s}|{p}"] = (s, int(commit_ts[s]), text)

H = H.assign(history_id=H.sha + "|" + H.path_in_commit, ts=H.sha.map(commit_ts))
H = H.sort_values(["row_id", "ts", "sha"], ascending=[True, False, False])
ROWS["history_ids"] = ROWS.row_id.map(H.groupby("row_id").history_id.agg(list)).map(
    lambda v: v if isinstance(v, list) else [])

late = int((H.ts > H.row_id.map(dict(zip(ROWS.row_id, ROWS.cutoff_ts)))).sum())
per_row = ROWS.history_ids.map(len)
REPORT.update({"bot commits left out": len(bots & set(HC.sha)), "history texts": len(HIST), **how,
               "commit lines dated after their release (kept: ancestry decides)": late})
print(f"bot commits left out {len(bots & set(HC.sha)):,}   history texts {len(HIST):,}")
for k, v in how.items():
    print(f"   {k:<46}{v:>9,} ({v / max(1, len(HIST)):.1%})")
print(f"commit lines dated after their release (kept, git ancestry is the rule) {late:,}")
print(f"commits per row: median {per_row.median():.0f}   90th percentile {per_row.quantile(.9):.0f}   "
      f"maximum {per_row.max():,}   rows with none {int((per_row == 0).sum()):,}")

bot commits left out 1   history texts 70,688
   message + diff                                   70,428 (99.6%)
   message only: no changed lines for this file        260 (0.4%)
commit lines dated after their release (kept, git ancestry is the rule) 6
commits per row: median 8   90th percentile 27   maximum 577   rows with none 3


### Cell 7 — Discourse texts: one per body, comment or review
Each issue or PR linked to a row (Notebook 2) gives its body and each of its comments and reviews as
separate texts, each with its own time. Bot-written and empty texts are left out. A row gets only the
texts **written before its release's cutoff**, newest first.

In [18]:
body = ITEMS[~ITEMS.bot.astype(bool) & (ITEMS.text.fillna("").str.strip() != "")]
talk = COMMENTS[~COMMENTS.bot.astype(bool) & (COMMENTS.text.fillna("").str.strip() != "")]
UNITS = pd.concat([
    pd.DataFrame({"discourse_id": REPO + "|" + body.kind + "|" + body.number.astype(str) + "|body",
                  "item_kind": body.kind, "number": body.number, "kind": body.kind + " body",
                  "ts": body.ts, "text": body.text}),
    pd.DataFrame({"discourse_id": REPO + "|" + talk.kind + "|" + talk.number.astype(str) + "|" + talk.source
                                  + "|" + talk.record_id.astype(str),
                  "item_kind": talk.kind, "number": talk.number,
                  "kind": np.where(talk.source == "review", "review", talk.kind + " comment"),
                  "ts": talk.ts, "text": talk.text})], ignore_index=True)
if UNITS.discourse_id.duplicated().any():
    raise SystemExit("a discourse id repeats")

D = LINKS.merge(ROWS[["row_id", "cutoff_ts"]], on="row_id") \
         .merge(UNITS[["discourse_id", "item_kind", "number", "ts"]],
                left_on=["kind", "number"], right_on=["item_kind", "number"])
D = D[D.ts < D.cutoff_ts].sort_values(["row_id", "ts", "discourse_id"], ascending=[True, False, False])
ROWS["discourse_ids"] = ROWS.row_id.map(D.groupby("row_id").discourse_id.agg(list)).map(
    lambda v: v if isinstance(v, list) else [])
UNITS = UNITS[UNITS.discourse_id.isin(set(D.discourse_id))]

per_row = ROWS.discourse_ids.map(len)
REPORT.update({"bot or empty texts left out": int(len(ITEMS) + len(COMMENTS) - len(body) - len(talk)),
               "discourse texts": len(UNITS)})
print(f"discourse texts used {len(UNITS):,}   " + "   ".join(f"{k} {v:,}" for k, v in UNITS.kind.value_counts().items()))
print(f"texts per row: median {per_row.median():.0f}   90th percentile {per_row.quantile(.9):.0f}   "
      f"maximum {per_row.max():,}   rows with none {int((per_row == 0).sum()):,}")

discourse texts used 23,153   review 13,498   pr comment 5,126   pr body 2,621   issue comment 1,268   issue body 640
texts per row: median 70   90th percentile 358   maximum 6,796   rows with none 3,966


### Cell 8 — Check, write, and the manifest
Stops if any of these fails, before anything is written:
- every row has a split, a label of 0 or 1, and at least one code text;
- every id a row lists exists in its text table, and no text is empty;
- no discourse text a row lists is dated at or after the row's cutoff;
- the label appears in no text table.

Then writes the five files and `manifest_<repo>.json` (each file's lines and SHA-256, and this run's
report), ready for the Zenodo upload.

In [19]:
from importlib.metadata import version
code_df = pd.DataFrame([(k, kind, t) for k, (kind, t) in CODE.items()], columns=["code_id", "kind", "text"])
hist_df = pd.DataFrame([(k, s, ts, t) for k, (s, ts, t) in HIST.items()],
                       columns=["history_id", "sha", "commit_ts", "text"])
disc_df = UNITS[["discourse_id", "kind", "ts", "text"]].reset_index(drop=True)
rows_df = ROWS[["row_id", "repo", "release", "release_pos", "path", "cutoff_ts", "label", "split",
                "code_ids", "history_ids", "discourse_ids"]]

listed = lambda col: {i for ids in rows_df[col] for i in ids}
disc_ts = dict(zip(disc_df.discourse_id, disc_df.ts))
problems = {
    "rows without a split": int(rows_df.split.isna().sum()),
    "labels other than 0/1": int((~rows_df.label.isin([0, 1])).sum()),
    "rows without code": int((rows_df.code_ids.map(len) == 0).sum()),
    "code ids missing": len(listed("code_ids") - set(code_df.code_id)),
    "history ids missing": len(listed("history_ids") - set(hist_df.history_id)),
    "discourse ids missing": len(listed("discourse_ids") - set(disc_df.discourse_id)),
    "empty texts": int(sum((df.text.str.strip() == "").sum() for df in (code_df, hist_df, disc_df))),
    "discourse dated at or after the cutoff": int(sum(disc_ts[i] >= c for ids, c in
                                                      zip(rows_df.discourse_ids, rows_df.cutoff_ts) for i in ids)),
    "label in a text table": int(any("label" in df.columns for df in (code_df, hist_df, disc_df))),
}
bad = {k: v for k, v in problems.items() if v}
if bad:
    raise SystemExit(f"{REPO}: {bad}")
print("all checks passed\n")

files = {f"rows_{REPO}.parquet": rows_df, f"code_texts_{REPO}.parquet": code_df,
         f"history_texts_{REPO}.parquet": hist_df, f"discourse_texts_{REPO}.parquet": disc_df}
for name, df in files.items():
    df.to_parquet(OUT / name, index=False)
S[["release", "release_pos", "cutoff", "label_known", "part"]].to_csv(OUT / f"split_{REPO}.csv", index=False)

manifest = {"notebook": "notebook_3_dataset_CBERT", "repo": REPO, "split": SPLIT,
            "versions": {p: version(p) for p in ("tree-sitter", "tree-sitter-java", "pandas", "pyarrow")},
            "files": [{"file": name, "lines": len(df) if name in files else len(S),
                       "sha256": hashlib.sha256((OUT / name).read_bytes()).hexdigest()}
                      for name, df in {**files, f"split_{REPO}.csv": S}.items()],
            "report": {k: (int(v) if isinstance(v, (np.integer, int)) else v) for k, v in REPORT.items()}}
(OUT / f"manifest_{REPO}.json").write_text(json.dumps(manifest, indent=1), encoding="utf-8")
for f in manifest["files"]:
    print(f"{f['file']:<36}{f['lines']:>10,} lines")
print(f"\nwrote {len(manifest['files'])} files and manifest_{REPO}.json to {OUT}")

all checks passed

rows_questdb.parquet                   218,893 lines
code_texts_questdb.parquet             113,077 lines
history_texts_questdb.parquet           70,688 lines
discourse_texts_questdb.parquet         23,153 lines
split_questdb.csv                          114 lines

wrote 5 files and manifest_questdb.json to E:\Thesis Project\Smell Ranker\workspace_data\outputs\notebook_3_cbert


### Cell 9 — Summary of all five repositories
Run once all five repositories have been through cells 1a–8. It reads their outputs and prints the
numbers the dataset's description reports: rows, texts, the split, discourse texts by kind and texts
per row. It changes nothing.

In [6]:
# DATASET SUMMARY: every number the Zenodo description and the chapter need, read from this
# notebook's outputs for all five repositories. It changes nothing.
REPOS = ["commons-lang", "checkstyle", "dubbo", "junit-framework", "questdb"]
NOTE = {"message only: commit not in the diffs file": "msg only, commit not in diffs",
        "message only: no changed lines for this file": "msg only, no changed line",
        "file versions not parsed (kept whole)": "kept whole",
        "commit lines dated after their release (kept: ancestry decides)": "late commit lines"}
per_repo, per_part, kinds, per_row = [], [], [], []
for repo in REPOS:
    D = OUT.parent / repo                            # notebook_3_cbert/<repo>
    rows = pd.read_parquet(D / f"rows_{repo}.parquet")
    code = pd.read_parquet(D / f"code_texts_{repo}.parquet", columns=["kind"])
    hist = pd.read_parquet(D / f"history_texts_{repo}.parquet", columns=["history_id"])
    disc = pd.read_parquet(D / f"discourse_texts_{repo}.parquet", columns=["kind"])
    rep = json.loads((D / f"manifest_{repo}.json").read_text(encoding="utf-8"))["report"]
    labelled = rows[rows.split != "left out"]
    per_repo.append({"repo": repo, "releases": rows.release.nunique(), "rows": len(rows),
                     "refactored": int(rows.label.sum()), "code texts": len(code),
                     "history texts": len(hist), "discourse texts": len(disc),
                     "kind=file": int((code.kind == "file").sum()),
                     **{short: rep.get(long, 0) for long, short in NOTE.items()},
                     "rows no history": int((rows.history_ids.map(len) == 0).sum()),
                     "rows no discourse": int((rows.discourse_ids.map(len) == 0).sum())})
    for part, g in rows.groupby("split"):
        per_part.append({"repo": repo, "part": part, "releases": g.release.nunique(),
                         "rows": len(g), "refactored": int(g.label.sum())})
    kinds.append(disc.kind.value_counts().rename(repo))
    for view in ("code_ids", "history_ids", "discourse_ids"):
        n = rows[view].map(len) - (1 if view == "code_ids" else 0)      # code: methods only
        per_row.append({"repo": repo, "view": view[:-4], "median": int(n.median()),
                        "90th": int(n.quantile(.9)), "max": int(n.max()), "none": int((n == 0).sum())})

A = pd.DataFrame(per_repo); A.loc[len(A)] = ["All"] + list(A.drop(columns="repo").sum())
print("PER REPO\n" + A.to_string(index=False))
P = pd.DataFrame(per_part)
print("\nPER PART, PER REPO\n" + P.to_string(index=False))
T = P.groupby("part")[["releases", "rows", "refactored"]].sum()
T["share"] = (T.refactored / T.rows).round(4)
print("\nPER PART, ALL\n" + T.to_string())
print("\nDISCOURSE TEXTS BY KIND\n" + pd.concat(kinds, axis=1).fillna(0).astype(int).to_string())
print("\nTEXTS PER ROW (code = methods)\n" + pd.DataFrame(per_row).to_string(index=False))

PER REPO
           repo  releases   rows  refactored  code texts  history texts  discourse texts  kind=file  msg only, commit not in diffs  msg only, no changed line  kept whole  late commit lines  rows no history  rows no discourse
   commons-lang        28   4124         793       19476           9781             2225          0                              5                         79           0                  0                0               2270
     checkstyle       201  70105        3640       30094          25981            55855          0                              1                       1434           0                  0               48               5340
          dubbo       106 140453        5805       58976          34027            18214          2                            908                       1081           2                  0                0              16721
junit-framework        48  28545        1590       14204          23880            16091   